## Complex model for selecting best candidate

1. on input: data from NER and 10 candidates for each named entity, and ground truth entities with targt DBpedia URI - this comes from trainig dataset
2. Then train the model, and perform NER.
3. on output: metric showing how well the model selects the candidates - on the training dataset, and then on test dataset


## Import libraries

In [76]:
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from datetime import datetime
from sentence_transformers import SentenceTransformer, util, CrossEncoder
from fuzzywuzzy import fuzz
from nltk.tokenize import RegexpTokenizer
from concurrent.futures import ThreadPoolExecutor, as_completed

## Model with 6 features - training and evaluation

### Data preparation

In [96]:
# aida dataset train data
json_file_path = "../DjangoApp/NEL_project/NEL_app/Evaluation/candidate_selection_data/candidates_data_collector_20250424_104609.json" # aida_train_golden_annotations_with_candidates.json"

In [104]:
# Load the JSON data
with open(json_file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

# Show top-level structure
print("Keys in loaded JSON:")
print(data.keys())

# Access configuration
print("\nConfiguration:")
print(data["configuration"])

# Access one golden annotation entry
print("\nSample Golden Annotation:")
print(json.dumps(data["named_entities_annotations"][0], indent=2))

# Access one prediction entry and its candidates
print("\nSample Prediction with Candidates:")
print(json.dumps(data["predictions"][0], indent=2))

golden_annotations = data["named_entities_annotations"]
predictions = data["predictions"]

Keys in loaded JSON:
dict_keys(['name', 'configuration', 'named_entities_annotations', 'predictions'])

Configuration:
{'dataset_path': './EvaluationDatasets/aida_train_converted.json', 'dataset_total_texts': 940, 'dataset_total_mentions': 18395, 'ned_knowledge_graph': 'dbpedia', 'use_golden_annotations': True, 'ner_model_name': None, 'ner_accuracy': 'N/A'}

Sample Golden Annotation:
{
  "content": "EU rejects German call to boycott British lamb . Peter Blackburn BRUSSELS 1996-08-22 The European Commission said on Thursday it disagreed with German advice to consumers to shun British lamb until scientists determine whether mad cow disease can be transmitted to sheep . Germany 's representative to the European Union 's veterinary committee Werner Zwingmann said on Wednesday consumers should buy sheepmeat from countries other than Britain until the scientific advice was clearer . \" We do n't support any such recommendation because we do n't see any grounds for it , \" the Commission 's c

### Candidates Scorers

In [166]:
import torch
from sentence_transformers import SentenceTransformer, util, CrossEncoder
from fuzzywuzzy import fuzz
import numpy as np
from nltk.tokenize import RegexpTokenizer
import pandas as pd
from datetime import datetime


# Define scorers:
class ContextScorerBi:
    def __init__(self, model_name="sentence-transformers/all-mpnet-base-v2", round_to=3, device='cuda'):
        self.device = device if torch.cuda.is_available() else 'cpu'
        self.model = SentenceTransformer(model_name).to(self.device)
        self.round_to = round_to

    def score(self, context_text, candidate_comment):
        context_emb = self.model.encode(context_text, convert_to_tensor=True)
        candidate_emb = self.model.encode(candidate_comment, convert_to_tensor=True)
        score = util.pytorch_cos_sim(context_emb, candidate_emb).item()
        return round(score, self.round_to)

class LevenshteinDistanceScorer:
    def __init__(self, round_to=3):
        self.round_to = round_to

    def score(self, label1, label2):
        # Using token_sort_ratio for better handling of word order variations
        return round(fuzz.ratio(label1, label2) / 100.0, self.round_to)
        
class PopularityScorer:
    def __init__(self, round_to=3):
        self.round_to = round_to
        self.min_log_ref_count = None
        self.max_log_ref_count = None

    def fit(self, candidates):
        """Calculates min and max log popularity scores for normalization."""
        ref_counts = [c.get("ref_count", 0) for c in candidates]
        log_counts = np.log1p(ref_counts)
        self.min_log_ref_count = np.min(log_counts) if log_counts.size > 0 else 0
        self.max_log_ref_count = np.max(log_counts) if log_counts.size > 0 else 1 # Avoid division by zero

    def score(self, candidate):
        """Scores a single candidate based on popularity."""
        ref_count = candidate.get("ref_count", 0)
        log_val = np.log1p(ref_count)
        if self.max_log_ref_count == self.min_log_ref_count:
            return round(1.0, self.round_to)
        else:
            normalized = (log_val - self.min_log_ref_count) / (self.max_log_ref_count - self.min_log_ref_count)
            return round(normalized, self.round_to)

    def score_all(self, candidates):
        """Scores all candidates (using the fit method for normalization)."""
        self.fit(candidates)
        return [self.score(c) for c in candidates]

class DbpediaCandidatePositionScorer:
    def __init__(self, round_to=3, max_candidates=10):
        """
        Initializes the DbpediaCandidatePositionScorer.

        Args:
            round_to (int): The number of decimal places to round the score to.
            max_candidates (int): The maximum number of candidates expected.
        """
        self.round_to = round_to
        self.max_candidates = max_candidates

    def score_all(self, candidates):
        """
        Returns a list of raw (1-based) indices (positions) for all candidates.

        Args:
            candidates (list): A list of candidate objects.

        Returns:
            list: A list of 1-based indices corresponding to each candidate's position.
        """
        raw_scores = []
        for i, _ in enumerate(candidates):
            raw_scores.append(i + 1)  # Candidate object not needed for this score
        return raw_scores

    def normalize_score(self, raw_scores):
        """
        Inverts and normalizes the raw position scores to the range [0, 1].

        Args:
            raw_scores (list): A list of raw (1-based) position scores.

        Returns:
            list: A list of normalized position scores.
        """
        if not raw_scores:
            return []

        df = pd.DataFrame({'score_position': raw_scores})
        min_pos = df['score_position'].min()  # Should be 1
        max_pos = df['score_position'].max()  # Should be the number of candidates

        # Handle the case where there's only one candidate
        if max_pos == min_pos:
            normalized_scores = pd.Series([1.0] * len(raw_scores))
        else:
            normalized_scores = (max_pos - df['score_position']) / (max_pos - min_pos)

        return normalized_scores.round(self.round_to).tolist()

# ========== Initialize Scorers ==========
context_scorer_bi = ContextScorerBi()
levenshtein_scorer = LevenshteinDistanceScorer()
popularity_scorer = PopularityScorer()
position_scorer = DbpediaCandidatePositionScorer()


### Scoring of candidates

In [172]:
def score_candidate_predictions(predictions, dataset_name="Dataset"):
    """
    Scores candidates for each entity mention in the predictions using multiple scoring strategies.
    
    Parameters:
        predictions (List[Dict]): A list of annotated entries with entity candidates.
        dataset_name (str): Optional name of the dataset (for logging).
    
    Returns:
        pd.DataFrame: Scored DataFrame with features for each candidate.
    """
    scored_data = []

    for entry in tqdm(predictions, desc=f"Scoring predictions for {dataset_name}"):
        context_text = entry["content"]
        for entity in entry["entities"]:
            entity_label = entity["entity_label"]
            start_position = entity["start_position"]
            end_position = entity["end_position"]
            candidates = entity["candidates"]

            # Batch scoring
            pop_scores = popularity_scorer.score_all(candidates)
            pos_scores_raw = list(range(1, len(candidates) + 1))
            normalized_pos_scores = position_scorer.normalize_score(pos_scores_raw)

            for i, candidate in enumerate(candidates):
                label = candidate["label"]
                comment = candidate.get("comment", "")

                score_lev = levenshtein_scorer.score(entity_label, label)
                score_ctx_bi = context_scorer_bi.score(context_text, comment)
                score_pop = pop_scores[i]
                score_pos = normalized_pos_scores[i]

                scored_data.append({
                    "context_text": context_text,
                    "entity_label": entity_label,
                    "start_position": start_position,
                    "end_position": end_position,
                    "candidate_label": label,
                    "candidate_uri": candidate["uri"],
                    "score_levenshtein": score_lev,
                    "score_context_bi": score_ctx_bi,
                    "score_popularity": score_pop,
                    "score_position": score_pos,
                })

    df = pd.DataFrame(scored_data)
    return df

In [193]:
train_df = score_candidate_predictions(predictions, dataset_name="AIDA Training")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

csv_filename = f'aida_train_golden_annotations_with_candidates_scored_data_{timestamp}.csv' # changed!
train_df.to_csv(csv_filename, index=False)

print(f"DataFrame saved to {csv_filename}")

Scoring predictions for AIDA Training:   0%|          | 0/940 [00:02<?, ?it/s]


KeyboardInterrupt: 

### Get scored candidates from csv

In [177]:
train_df = pd.read_csv("scored_data_20250424_112539.csv")
train_df.drop(columns=["score_dbpedia"], inplace=True)

In [178]:
train_df.head()

,context_text,entity_label,start_position,end_position,candidate_label,candidate_uri,score_levenshtein,score_context_bi,score_popularity,score_position
0,EU rejects German call to boycott British lamb...,German,11,17,German Empire,http://dbpedia.org/resource/German_Empire,0.63,0.072,0.566,1.000
1,EU rejects German call to boycott British lamb...,German,11,17,German language,http://dbpedia.org/resource/German_language,0.57,0.111,0.577,0.889
2,EU rejects German call to boycott British lamb...,German,11,17,Germany,http://dbpedia.org/resource/Germany,0.92,0.175,1.000,0.778
3,EU rejects German call to boycott British lamb...,German,11,17,West Germany,http://dbpedia.org/resource/West_Germany,0.67,0.098,0.637,0.667
4,EU rejects German call to boycott British lamb...,German,11,17,Nazi Germany,http://dbpedia.org/resource/Nazi_Germany,0.67,0.100,0.546,0.556


In [179]:
train_df.describe()

,start_position,end_position,score_levenshtein,score_context_bi,score_popularity,score_position
count,183868.000000,183868.000000,183868.000000,183868.000000,183868.000000,183868.000000
mean,823.993947,832.981791,0.472527,0.188663,0.381425,0.500000
std,899.588983,899.728480,0.270146,0.176217,0.339219,0.319278
min,0.000000,3.000000,0.000000,-0.208000,0.000000,0.000000
25%,212.000000,221.000000,0.260000,0.042000,0.075000,0.222000
50%,544.000000,553.000000,0.460000,0.160000,0.293000,0.500000
75%,1096.000000,1105.000000,0.650000,0.317000,0.638000,0.778000
max,6887.000000,6895.000000,1.000000,0.843000,1.000000,1.000000


In [180]:

# ========== Prepare Data for the Neural Network with Grouping ==========

def create_best_candidate_flag_grouped(predictions_df, golden_annotations):
    best_candidate_uris = {}
    for gold_entry in golden_annotations:
        for entity in gold_entry['entities']:
            key = (gold_entry['content'], entity['entity_label'], entity['start_position'], entity['end_position'])
            best_candidate_uris[key] = entity['best_candidate_uri']

    predictions_df['is_best_candidate'] = False
    for index, row in predictions_df.iterrows():
        key = (row['context_text'], row['entity_label'], row['start_position'], row['end_position'])
        if key in best_candidate_uris and row['candidate_uri'] == best_candidate_uris[key]:
            predictions_df.loc[index, 'is_best_candidate'] = True
    return predictions_df

train_df_with_best_flag = create_best_candidate_flag_grouped(train_df.copy(), golden_annotations)


In [181]:
train_df_with_best_flag.head()

,context_text,entity_label,start_position,end_position,candidate_label,candidate_uri,score_levenshtein,score_context_bi,score_popularity,score_position,is_best_candidate
0,EU rejects German call to boycott British lamb...,German,11,17,German Empire,http://dbpedia.org/resource/German_Empire,0.63,0.072,0.566,1.000,False
1,EU rejects German call to boycott British lamb...,German,11,17,German language,http://dbpedia.org/resource/German_language,0.57,0.111,0.577,0.889,False
2,EU rejects German call to boycott British lamb...,German,11,17,Germany,http://dbpedia.org/resource/Germany,0.92,0.175,1.000,0.778,True
3,EU rejects German call to boycott British lamb...,German,11,17,West Germany,http://dbpedia.org/resource/West_Germany,0.67,0.098,0.637,0.667,False
4,EU rejects German call to boycott British lamb...,German,11,17,Nazi Germany,http://dbpedia.org/resource/Nazi_Germany,0.67,0.100,0.546,0.556,False


### Random Forest Classifier - training and evaluation

#### Trainig

In [182]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Assuming df_with_best_flag is your DataFrame as described

# === 1. Build a grouped dataset for Random Forest ===
def build_grouped_dataframe_rf(df, max_candidates=10):
    grouped_features = []
    targets = []

    # group by unique entity occurrence
    # Metrics: ('levenshtein', 'context_bi', 'position', 'popularity')
    for key, group in df.groupby(['context_text','entity_label','start_position','end_position']):
        candidates = group[['score_levenshtein','score_context_bi', 'score_popularity', 'score_position']].values
        # pad up to max_candidates
        if len(candidates) < max_candidates:
            pad = np.zeros((max_candidates - len(candidates), candidates.shape[1]))
            candidates = np.vstack([candidates, pad])
        elif len(candidates) > max_candidates:
            candidates = candidates[:max_candidates]

        # find the index of the best
        best_idx = group['is_best_candidate'].values.argmax()  # first True
        grouped_features.append(candidates.flatten().astype(np.float32)) # Flatten for RF
        targets.append(best_idx)

    X = np.stack(grouped_features)      # (N_entities, 10 * 6)
    y = np.array(targets)                # (N_entities,)
    return X, y

X_rf, y_rf = build_grouped_dataframe_rf(train_df_with_best_flag, max_candidates=10)
print("X_rf shape:", X_rf.shape, "y_rf shape:", y_rf.shape)

# === 2. Train/Val Split for Random Forest ===
X_train_rf, X_val_rf, y_train_rf, y_val_rf = train_test_split(X_rf, y_rf, test_size=0.2)

# === 3. Define the Random Forest Pipeline ===
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),      # Feature scaling is often beneficial for RF
    ('classifier', RandomForestClassifier()) # You can add hyperparameters here
])

# === 4. Training the Random Forest Model ===
rf_pipeline.fit(X_train_rf, y_train_rf)

# === 5. Evaluation on Validation Set ===
y_pred_val_rf = rf_pipeline.predict(X_val_rf)
val_accuracy_rf = accuracy_score(y_val_rf, y_pred_val_rf)
print(f"\nValidation Accuracy (Random Forest): {val_accuracy_rf:.4f}")
print("\nValidation Classification Report (Random Forest):\n", classification_report(y_val_rf, y_pred_val_rf))

# === 6. Feature Importance Assessment ===
feature_names_rf = [
    f"cand_{i}_score_levenshtein" for i in range(10)
] + [
    f"cand_{i}_score_context_bi" for i in range(10)
] + [
    f"cand_{i}_score_popularity" for i in range(10)
] + [
    f"cand_{i}_score_position" for i in range(10)
] 

if hasattr(rf_pipeline.named_steps['classifier'], 'feature_importances_'):
    importances = rf_pipeline.named_steps['classifier'].feature_importances_
    feature_importance_dict = dict(zip(feature_names_rf, importances))
    sorted_feature_importance = sorted(feature_importance_dict.items(), key=lambda item: item[1], reverse=True)

    print("\nFeature Importance (Random Forest):")
    for feature, importance in sorted_feature_importance:
        print(f"{feature}: {importance:.4f}")

    # Identify potentially less important metrics (across all candidates)
    importance_per_metric = {}
    for i in range(10):
        importance_per_metric['levenshtein'] = importance_per_metric.get('levenshtein', 0) + feature_importance_dict[f"cand_{i}_score_levenshtein"]
        importance_per_metric['context_bi'] = importance_per_metric.get('context_bi', 0) + feature_importance_dict[f"cand_{i}_score_context_bi"]
        importance_per_metric['popularity'] = importance_per_metric.get('popularity', 0) + feature_importance_dict[f"cand_{i}_score_popularity"]
        importance_per_metric['position'] = importance_per_metric.get('position', 0) + feature_importance_dict[f"cand_{i}_score_position"]

    sorted_metric_importance = sorted(importance_per_metric.items(), key=lambda item: item[1], reverse=True)
    print("\nTotal Importance per Metric:")
    for metric, total_importance in sorted_metric_importance:
        print(f"{metric}: {total_importance:.4f}")

    # Suggest dropping less important metrics (you'll need to define a threshold)
    threshold = 0.01 # Example threshold - adjust as needed
    less_important_metrics = [metric for metric, importance in sorted_metric_importance if importance < threshold]
    if less_important_metrics:
        print(f"\nPotentially less important metrics (below threshold {threshold}): {less_important_metrics}")
        print("Consider removing these metrics from your feature set and retraining.")
    else:
        print("\nAll metrics appear to have some level of importance based on the current model.")
else:
    print("\nFeature importance is not available for this Random Forest model.")



X_rf shape: (18394, 40) y_rf shape: (18394,)

Validation Accuracy (Random Forest): 0.9361

Validation Classification Report (Random Forest):
               precision    recall  f1-score   support

           0       0.94      0.99      0.96      2843
           1       0.92      0.78      0.84       313
           2       0.94      0.86      0.90       195
           3       0.95      0.72      0.82        80
           4       0.96      0.65      0.78        78
           5       0.93      0.68      0.79        63
           6       0.97      0.74      0.84        43
           7       0.92      0.63      0.75        19
           8       0.81      0.65      0.72        20
           9       0.86      0.76      0.81        25

    accuracy                           0.94      3679
   macro avg       0.92      0.75      0.82      3679
weighted avg       0.94      0.94      0.93      3679


Feature Importance (Random Forest):
cand_1_score_levenshtein: 0.0770
cand_5_score_levenshtein: 0.0

#### Evaluation with test dataset - AIDA test

##### Helper functions

In [190]:
import pandas as pd
import json
from sklearn.metrics import accuracy_score, classification_report

# === 1. Define Reusable Helper Functions ===

def load_golden_and_predictions(filepath):
    with open(filepath, "r", encoding="utf-8") as file:
        data = json.load(file)
    return data["golden_annotations"], data["predictions"]
    
def describe_dataset(name, df):
    print(f"\n\n===== Dataset Description: {name} =====")
    print(df.describe())  # include='all' for mixed types

def create_best_candidate_flag_grouped(predictions_df, golden_annotations):
    best_candidate_uris = {}
    for gold_entry in golden_annotations:
        for entity in gold_entry['entities']:
            key = (gold_entry['content'], entity['entity_label'], entity['start_position'], entity['end_position'])
            best_candidate_uris[key] = entity['best_candidate_uri']

    predictions_df['is_best_candidate'] = False
    for index, row in predictions_df.iterrows():
        key = (row['context_text'], row['entity_label'], row['start_position'], row['end_position'])
        if key in best_candidate_uris and row['candidate_uri'] == best_candidate_uris[key]:
            predictions_df.at[index, 'is_best_candidate'] = True
    return predictions_df

def evaluate_dataset(name, df_scored, golden_annotations, rf_pipeline, max_candidates=10):
    print(f"\n===== Evaluating Dataset: {name} =====")

    # Flag best candidates
    df_flagged = create_best_candidate_flag_grouped(df_scored.copy(), golden_annotations)

    # Extract features and labels
    X_test, y_test = build_grouped_dataframe_rf(df_flagged, max_candidates=max_candidates)

    # Predict and evaluate
    y_pred = rf_pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    print(f"\n{name} - Test Accuracy (Random Forest): {accuracy:.4f}")
    print(f"{name} - Classification Report:\n", classification_report(y_test, y_pred))

    return accuracy



##### Import dataset

In [184]:
test_json_file_path_aida = "../DjangoApp/NEL_project/NEL_app/Evaluation/candidate_selection_data/aida_test_golden_annotations_with_candidates.json"
test_json_file_path_ace = "../DjangoApp/NEL_project/NEL_app/Evaluation/candidate_selection_data/ace_test_golden_annotations_with_candidates.json"

test_golden_annotations_aida, test_predictions_aida = load_golden_and_predictions(test_json_file_path_aida)
test_golden_annotations_ace, test_predictions_ace = load_golden_and_predictions(test_json_file_path_ace)


#### Calcualte scores using the original json file

In [152]:
test_df_aida = score_candidate_predictions(test_predictions_aida, dataset_name="AIDA Test")
test_df_ace = score_candidate_predictions(test_predictions_ace, dataset_name="ACE Test")

csf_filename_aida = "test_aida_golden_annotations_scored.csv"
csf_filename_ace = "test_ace_golden_annotations_scored.csv"

test_df_aida.to_csv(csf_filename_aida, index=False)
print(f"DataFrame saved to {csf_filename_aida}")

test_df_ace.to_csv(csf_filename_ace, index=False)
print(f"DataFrame saved to {csf_filename_ace}")


Scoring predictions for ACE Test: 100%|██████████| 119/119 [00:21<00:00,  5.60it/s]


DataFrame saved to test_aida_golden_annotations_scored.csv
DataFrame saved to test_ace_golden_annotations_scored.csv


#### Import scored candidaes from csv

In [187]:
# === 2. Load Datasets ===
test_df_aida = pd.read_csv("test_aida_golden_annotations_scored.csv")
test_df_aida.drop(columns=["score_dbpedia"], inplace=True)
test_df_ace = pd.read_csv("test_ace_golden_annotations_scored.csv")
test_df_ace.drop(columns=["score_dbpedia"], inplace=True)

In [191]:
# === 3. Describe Datasets ===
describe_dataset("AIDA", test_df_aida)
describe_dataset("ACE", test_df_ace)



===== Dataset Description: AIDA =====
       start_position  end_position  score_levenshtein  score_context_bi  \
count    44610.000000  44610.000000       44610.000000      44610.000000   
mean       730.298700    739.195091           0.460290          0.198159   
std        695.540818    695.532576           0.272301          0.169520   
min          0.000000      3.000000           0.000000         -0.170000   
25%        208.000000    218.000000           0.240000          0.062000   
50%        523.000000    532.000000           0.440000          0.180000   
75%       1059.000000   1065.000000           0.640000          0.310000   
max       4015.000000   4045.000000           1.000000          0.811000   

       score_popularity  score_position  
count      44610.000000    44610.000000  
mean           0.378351        0.500011  
std            0.339316        0.319268  
min            0.000000        0.000000  
25%            0.073000        0.222000  
50%            0.289000

In [192]:
# === 4. Evaluate Both Datasets ===
evaluate_dataset("AIDA", test_df_aida, test_golden_annotations_aida, rf_pipeline, max_candidates=10)
evaluate_dataset("ACE", test_df_ace, test_golden_annotations_ace, rf_pipeline, max_candidates=10)


===== Evaluating Dataset: AIDA =====

AIDA - Test Accuracy (Random Forest): 0.8371
AIDA - Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.99      0.91      3359
           1       0.89      0.48      0.62       416
           2       0.88      0.34      0.49       256
           3       0.89      0.41      0.56       143
           4       0.60      0.14      0.23        84
           5       0.88      0.18      0.30        38
           6       0.47      0.16      0.23        58
           7       1.00      0.34      0.51        47
           8       1.00      0.34      0.51        32
           9       0.80      0.14      0.24        29

    accuracy                           0.84      4462
   macro avg       0.82      0.35      0.46      4462
weighted avg       0.84      0.84      0.81      4462


===== Evaluating Dataset: ACE =====

ACE - Test Accuracy (Random Forest): 0.8366
ACE - Classification Report:
               

/home/student/miniconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/student/miniconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/student/miniconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


0.8365758754863813

### Feature selection

In [36]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# === 1. Build a grouped dataset for Random Forest ===
def build_grouped_dataframe_rf(df, selected_metrics, max_candidates=10):
    grouped_features = []
    targets = []

    for key, group in df.groupby(['context_text', 'entity_label', 'start_position', 'end_position']):
        candidates = group[[f'score_{m}' for m in selected_metrics]].values
        if len(candidates) < max_candidates:
            pad = np.zeros((max_candidates - len(candidates), candidates.shape[1]))
            candidates = np.vstack([candidates, pad])
        elif len(candidates) > max_candidates:
            candidates = candidates[:max_candidates]

        best_idx = group['is_best_candidate'].values.argmax()
        grouped_features.append(candidates.flatten().astype(np.float32))
        targets.append(best_idx)

    X = np.stack(grouped_features)
    y = np.array(targets)
    return X, y

# === 2. Prepare Train and Final Test Data ===
all_metrics = ['levenshtein', 'context_bi', 'context_cross', 'jaccard', 'popularity', 'position']

# Split your main dataset into training only
df_train_split = df_with_best_flag  # Your full training dataset
df_test_final = test_df_with_best_flag  # Your separate, final test set

best_result = {'metrics': None, 'accuracy': 0.0, 'report': '', 'pipeline': None}

print("\nPerforming Feature Selection and Final Test Evaluation:\n")

for r in range(2, len(all_metrics) + 1):
    for metric_subset in combinations(all_metrics, r):
        print(f"Evaluating metric combination: {metric_subset}")

        # Prepare data using the current metric subset
        X_train_rf, y_train_rf = build_grouped_dataframe_rf(df_train_split, selected_metrics=metric_subset)
        X_test_rf, y_test_rf = build_grouped_dataframe_rf(df_test_final, selected_metrics=metric_subset)

        rf_pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', RandomForestClassifier(random_state=42))
        ])

        # Train on training data
        rf_pipeline.fit(X_train_rf, y_train_rf)

        # Evaluate on test data
        y_pred_rf = rf_pipeline.predict(X_test_rf)
        acc = accuracy_score(y_test_rf, y_pred_rf)
        print(f"  → Test Accuracy: {acc:.4f}\n")

        # Update best result if this combination is better
        if acc > best_result['accuracy']:
            best_result['metrics'] = metric_subset
            best_result['accuracy'] = acc
            best_result['report'] = classification_report(y_test_rf, y_pred_rf, digits=4)
            best_result['pipeline'] = rf_pipeline

# === 3. Final Results ===
print("\nBest Feature Combination Found (based on test set):")
print(f"Metrics: {best_result['metrics']}")
print(f"Test Accuracy: {best_result['accuracy']:.4f}")
print("\nDetailed Classification Report:\n")
print(best_result['report'])



Performing Feature Selection and Final Test Evaluation:

Evaluating metric combination: ('levenshtein', 'context_bi')
  → Test Accuracy: 0.8427

Evaluating metric combination: ('levenshtein', 'context_cross')
  → Test Accuracy: 0.8162

Evaluating metric combination: ('levenshtein', 'jaccard')
  → Test Accuracy: 0.8187

Evaluating metric combination: ('levenshtein', 'popularity')
  → Test Accuracy: 0.8182

Evaluating metric combination: ('levenshtein', 'position')
  → Test Accuracy: 0.8162

Evaluating metric combination: ('context_bi', 'context_cross')
  → Test Accuracy: 0.8104

Evaluating metric combination: ('context_bi', 'jaccard')
  → Test Accuracy: 0.8339

Evaluating metric combination: ('context_bi', 'popularity')
  → Test Accuracy: 0.8485

Evaluating metric combination: ('context_bi', 'position')
  → Test Accuracy: 0.8196

Evaluating metric combination: ('context_cross', 'jaccard')
  → Test Accuracy: 0.8019

Evaluating metric combination: ('context_cross', 'popularity')
  → Test

## Model with 7 featuers

#### NER types definitions

In [61]:
CLASS_DEFINITIONS = {
        "tomaarsen/span-marker-xlm-roberta-large-conllpp-doc-context": {
            "PER": "PER - Proper names of individuals, including first names, last names, fictional names, and unique nicknames.",
            "LOC": "LOC - Names of geographical locations such as cities, countries, states, provinces, and other physical locations.",
            "ORG": "ORG - Names of organizations including companies, government agencies, political parties, educational institutions, and sports teams.",
            "MISC": "MISC - Other named entities that do not fit into PER, LOC, or ORG categories, such as nationalities, artistic works, events, and other miscellaneous proper nouns.",
            "O": "O - Unknown"
        },
        "tomaarsen/span-marker-roberta-large-ontonotes5": {
            "PERSON": "PERSON - Proper names of people including first names, last names, individual or family names, fictional names and unique nicknames.",
            "NORP": "NORP - Adjectival forms of GPE and non-GPE place names (such as American), named religions, heritage, and political affiliation.",
            "FAC": "FAC - Names of man-made structures, including the buildings, airports, stations, infrastructures (bridges and streets), monuments, oil fields, golf courses, hospitals, zoos, shopping centers, etc.",
            "ORG": "ORG - Names of companies, government agencies, political parties, educational institutions, sport teams, hospitals, museums, libraries etc.",
            "GPE": "GPE - Names of geographical administrative entities including countries, villages, cities, states, provinces, prefectures, and other forms of municipalities",
            "LOC": "LOC - Names of locations other than GPEs including celestial bodies, stars, continents, mountains, oceans, coasts, rivers, lakes, borders, etc.",
            "PRODUCT": "PRODUCT - Name of any product including non-commercial vehicles (automobiles, rockets, aircraft, ships).",
            "EVENT": "EVENT - Named events and phenomena including natural disasters, hurricanes, revolutions, battles, wars, demonstrations, concerts, sports events, etc.",
            "WORK_OF_ART": "WORK OF ART - Titles of books, songs, films, plays and other creations such as awards, stock price indexes, and social security systems including health insurance systems or pension plans.",
            "LAW": "LAW - Named legal documents including laws, treaties, sections, and chapters.",
            "LANGUAGE": "LANGUAGE - Any named language including programming languages.",
            "O": "O - Unknown"
        },
        "tomaarsen/span-marker-bert-base-fewnerd-fine-super": {
            "art-broadcastprogram": "art-broadcastprogram - Broadcast programs including TV and radio shows.",
            "art-film": "art-film - Film titles and cinematic works.",
            "art-music": "art-music - Music-related works including performances, bands, and symphonies.",
            "art-other": "art-other - Other artistic creations not covered by film, music, painting, or written art.",
            "art-painting": "art-painting - Titles of paintings and art reproductions.",
            "art-writtenart": "art-writtenart - Literary and written art forms, including books and scripts.",
            "building-airport": "building-airport - Names of airports and aviation terminals.",
            "building-hospital": "building-hospital - Hospitals and medical facilities.",
            "building-hotel": "building-hotel - Hotels and lodging establishments.",
            "building-library": "building-library - Libraries and book repositories.",
            "building-other": "building-other - Other types of buildings not categorized elsewhere.",
            "building-restaurant": "building-restaurant - Restaurants and dining establishments.",
            "building-sportsfacility": "building-sportsfacility - Sports facilities and arenas.",
            "building-theater": "building-theater - Theaters and performance venues.",
            "event-attack/battle/war/militaryconflict": "event-attack/battle/war/militaryconflict - Military conflicts including attacks, battles, wars, and military engagements.",
            "event-disaster": "event-disaster - Natural or man-made disasters and catastrophic events.",
            "event-election": "event-election - Elections and voting events.",
            "event-other": "event-other - Other events not classified elsewhere.",
            "event-protest": "event-protest - Protests and demonstrations.",
            "event-sportsevent": "event-sportsevent - Sports events and competitions.",
            "location-GPE": "location-GPE - Geopolitical entities, typically countries, states, or cities.",
            "location-bodiesofwater": "location-bodiesofwater - Names of significant bodies of water such as lakes, rivers, and coasts.",
            "location-island": "location-island - Names of islands and archipelagos.",
            "location-mountain": "location-mountain - Mountain ranges, peaks, and related geographical features.",
            "location-other": "location-other - Other location names not covered by standard geographical categories.",
            "location-park": "location-park - Parks and recreational areas.",
            "location-road/railway/highway/transit": "location-road/railway/highway/transit - Transportation-related locations such as roads, railways, highways, and transit systems.",
            "organization-company": "organization-company - Companies and corporate entities.",
            "organization-education": "organization-education - Educational institutions and organizations.",
            "organization-government/governmentagency": "organization-government/governmentagency - Government bodies and agencies.",
            "organization-media/newspaper": "organization-media/newspaper - Media organizations and newspaper titles.",
            "organization-other": "organization-other - Other organizational entities not covered by other categories.",
            "organization-politicalparty": "organization-politicalparty - Political parties and related organizations.",
            "organization-religion": "organization-religion - Religious organizations and groups.",
            "organization-showorganization": "organization-showorganization - Organizations related to shows, entertainment, and performance groups.",
            "organization-sportsleague": "organization-sportsleague - Sports leagues and associations.",
            "organization-sportsteam": "organization-sportsteam - Sports teams and clubs.",
            "other-astronomything": "other-astronomything - Celestial objects and astronomy-related terms.",
            "other-award": "other-award - Awards and honors.",
            "other-biologything": "other-biologything - Biological terms and entities.",
            "other-chemicalthing": "other-chemicalthing - Chemical substances and compounds.",
            "other-currency": "other-currency - Currency symbols or names.",
            "other-disease": "other-disease - Diseases and medical conditions.",
            "other-educationaldegree": "other-educationaldegree - Academic degrees and certifications.",
            "other-god": "other-god - Deities and divine entities.",
            "other-language": "other-language - Language names or linguistic entities.",
            "other-law": "other-law - Legal terms and law-related entities.",
            "other-livingthing": "other-livingthing - Living organisms and fauna.",
            "other-medical": "other-medical - Medical specialties, professionals, or terms.",
            "person-actor": "person-actor - Actors and performers in film, television, or theater.",
            "person-artist/author": "person-artist/author - Artists and authors.",
            "person-athlete": "person-athlete - Athletes and sports figures.",
            "person-director": "person-director - Film or stage directors.",
            "person-other": "person-other - Other persons not categorized elsewhere.",
            "person-politician": "person-politician - Politicians and political figures.",
            "person-scholar": "person-scholar - Scholars, researchers, and academics.",
            "person-soldier": "person-soldier - Military personnel and soldiers.",
            "product-airplane": "product-airplane - Aircraft and airplanes.",
            "product-car": "product-car - Cars and automobiles.",
            "product-food": "product-food - Food items or brands.",
            "product-game": "product-game - Games or video game titles.",
            "product-other": "product-other - Other products or merchandise.",
            "product-ship": "product-ship - Ships and maritime vessels.",
            "product-software": "product-software - Software products or applications.",
            "product-train": "product-train - Trains and railway vehicles.",
            "product-weapon": "product-weapon - Weapons and armaments.",
            "O": "O - Unknown"
        }
}

#### New scorer - types embedding similarity

In [62]:
class TypesEmbeddingScorer:
    def __init__(self, ner_model_name=None, model_name="all-MiniLM-L6-v2", round_to_decimal_places=3, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.embeddings_model = SentenceTransformer(model_name).to(self.device)
        self.round_to_decimal_places = round_to_decimal_places
        self.ner_model_name = ner_model_name

        self.class_definitions = CLASS_DEFINITIONS.get(ner_model_name, {})
        self.ner_embeddings_cache = {
            ner_type: self.get_embedding(definition)
            for ner_type, definition in self.class_definitions.items()
            if ner_type != "O"
        }

    def get_embedding(self, sentence):
        return self.embeddings_model.encode(sentence, convert_to_tensor=True).to(self.device)

    def score_types_embedding(self, entity_probabilities: list, ontology_types: list) -> float:
        if not ontology_types or not entity_probabilities:
            return 0.0

        valid_ner_probabilities = [(cls, prob) for cls, prob in entity_probabilities if cls in self.ner_embeddings_cache]
        if not valid_ner_probabilities:
            return 0.0

        ner_classes = [item[0] for item in valid_ner_probabilities]
        ner_probabilities = np.array([float(item[1]) for item in valid_ner_probabilities])

        kg_embeddings = {kg_type: self.get_embedding(kg_type) for kg_type in ontology_types}
        similarity_matrix = np.zeros((len(ner_classes), len(ontology_types)))

        for i, ner_cls in enumerate(ner_classes):
            ner_embedding = self.ner_embeddings_cache[ner_cls]
            for j, kg_type in enumerate(ontology_types):
                kg_embedding = kg_embeddings.get(kg_type)
                if kg_embedding is not None:
                    similarity_matrix[i, j] = util.pytorch_cos_sim(ner_embedding.unsqueeze(0), kg_embedding.unsqueeze(0)).item()

        weighted_similarities = np.array([
            np.sum(ner_probabilities * similarity_matrix[:, j]) for j in range(len(ontology_types))
        ])
        return np.mean(weighted_similarities) if weighted_similarities.size > 0 else 0.0

    def normalize_score(self, raw_scores: list) -> list:
        if not raw_scores:
            return []
        min_score, max_score = min(raw_scores), max(raw_scores)
        if max_score == min_score:
            return [round(1.0, self.round_to_decimal_places)] * len(raw_scores)
        return [round((score - min_score) / (max_score - min_score), self.round_to_decimal_places) for score in raw_scores]

class TopKTypesEmbeddingScorer(TypesEmbeddingScorer):
    def __init__(self, top_k=3, **kwargs):
        super().__init__(**kwargs)
        self.top_k = top_k

    def score_types_embedding(self, entity_probabilities: list, ontology_types: list) -> float:
        if not ontology_types or not entity_probabilities:
            return 0.0

        valid_ner_probabilities = sorted(
            [(cls, prob) for cls, prob in entity_probabilities if cls in self.ner_embeddings_cache],
            key=lambda x: x[1],
            reverse=True
        )[:self.top_k]

        if not valid_ner_probabilities:
            return 0.0

        ner_classes = [item[0] for item in valid_ner_probabilities]
        ner_probabilities = np.array([float(item[1]) for item in valid_ner_probabilities])

        kg_embeddings = {kg_type: self.get_embedding(kg_type) for kg_type in ontology_types}
        similarity_matrix = np.zeros((len(ner_classes), len(ontology_types)))

        for i, ner_cls in enumerate(ner_classes):
            ner_embedding = self.ner_embeddings_cache[ner_cls]
            for j, kg_type in enumerate(ontology_types):
                kg_embedding = kg_embeddings.get(kg_type)
                if kg_embedding is not None:
                    similarity_matrix[i, j] = util.pytorch_cos_sim(ner_embedding.unsqueeze(0), kg_embedding.unsqueeze(0)).item()

        weighted_similarities = np.array([
            np.sum(ner_probabilities * similarity_matrix[:, j])
            for j in range(len(ontology_types))
        ])
        return float(np.max(weighted_similarities)) if weighted_similarities.size > 0 else 0.0


class MaxNERTypesEmbeddingScorer(TypesEmbeddingScorer):
    def score_types_embedding(self, entity_probabilities: list, ontology_types: list) -> float:
        if not ontology_types or not entity_probabilities:
            return 0.0

        valid_ner_probabilities = [(cls, prob) for cls, prob in entity_probabilities if cls in self.ner_embeddings_cache]
        if not valid_ner_probabilities:
            return 0.0

        ner_classes = [item[0] for item in valid_ner_probabilities]
        ner_probabilities = np.array([float(item[1]) for item in valid_ner_probabilities])

        kg_embeddings = {kg_type: self.get_embedding(kg_type) for kg_type in ontology_types}
        similarity_matrix = np.zeros((len(ner_classes), len(ontology_types)))

        for i, ner_cls in enumerate(ner_classes):
            ner_embedding = self.ner_embeddings_cache[ner_cls]
            for j, kg_type in enumerate(ontology_types):
                kg_embedding = kg_embeddings.get(kg_type)
                if kg_embedding is not None:
                    similarity_matrix[i, j] = util.pytorch_cos_sim(ner_embedding.unsqueeze(0), kg_embedding.unsqueeze(0)).item()

        max_similarities = np.max(similarity_matrix, axis=1)
        return float(np.sum(max_similarities * ner_probabilities))


#### Score candidates

In [70]:
import json
import os
import pandas as pd
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Assuming the scoring modules and process_entry are already defined and imported

# === Define Configurations ===
class NERDataConfiguration:
    def __init__(self, model_name, file_path):
        self.model_name = model_name
        self.file_path = file_path
        self.named_entities_annotations = []
        self.predictions = []

    def load(self):
        if not os.path.isfile(self.file_path):
            raise FileNotFoundError(f"File for '{self.model_name}' not found: {self.file_path}")
        with open(self.file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            self.named_entities_annotations = data.get("named_entities_annotations", [])
            self.predictions = data.get("predictions", [])

    def __repr__(self):
        return f"<NERDataConfiguration model={self.model_name} annotations={len(self.named_entities_annotations)} predictions={len(self.predictions)}>"


class MultiNERLoader:
    def __init__(self, config_paths: dict):
        self.configs = {
            model: NERDataConfiguration(model, os.path.abspath(path))
            for model, path in config_paths.items()
        }

    def load_all(self):
        for config in self.configs.values():
            try:
                config.load()
            except Exception as e:
                print(f"Failed to load {config.model_name}: {e}")

    def get_config(self, model_name: str) -> NERDataConfiguration:
        return self.configs.get(model_name)

    def get_all_models(self):
        return list(self.configs.keys())

In [71]:
config_paths = {
    "conll": "../DjangoApp/NEL_project/NEL_app/Evaluation/candidate_selection_data/aida_train_only identified and classified named entities with CONLL NER model.json",
    "ontonotes": "../DjangoApp/NEL_project/NEL_app/Evaluation/candidate_selection_data/aida_train_only identified and classified named entities with Ontonotes NER model.json",
    "fewnerd": "../DjangoApp/NEL_project/NEL_app/Evaluation/candidate_selection_data/aida_train_only identified and classified named entities with FewNERD NER model.json",
}

loader = MultiNERLoader(config_paths)
loader.load_all()


In [74]:
def process_entry(ner_annotation, prediction_entry):
    scored_candidates = []
    context_text = ner_annotation.get("content")
    ner_entities = ner_annotation.get("entities", [])
    prediction_entities = prediction_entry.get("entities", [])

    # Create a mapping between (start_position, end_position, entity_label) for NER and Predictions
    ner_entity_map = {
        (ent.get("start_position"), ent.get("end_position"), ent.get("entity_label")): ent
        for ent in ner_entities
    }

    for predicted_entity in prediction_entities:
        start = predicted_entity.get("start_position")
        end = predicted_entity.get("end_position")
        pred_label = predicted_entity.get("entity_label")
        ner_entity = ner_entity_map.get((start, end, pred_label))

        if ner_entity:
            entity_label = predicted_entity.get("entity_label")
            candidates = predicted_entity.get("candidates", [])
            entity_probabilities_raw = ner_entity.get("probabilities", [])
            entity_probabilities = []
            if entity_probabilities_raw:
                for prob in entity_probabilities_raw:
                    if len(prob) == 2:
                        entity_probabilities.append((prob[0], float(prob[1])))

            pop_scores = popularity_scorer.score_all(candidates)
            pos_scores = position_scorer.score_all(candidates)

            all_raw_pos_scores_local = []
            all_raw_types_embedding_scores_local_basic = []
            all_raw_types_embedding_scores_local_topk = []
            all_raw_types_embedding_scores_local_maxner = []

            scored_candidates_local = []

            for i, candidate in enumerate(candidates):
                label = candidate.get("label")
                comment = candidate.get("comment", "")
                ontology_types = candidate.get("ontology_types", [])

                score_lev = levenshtein_scorer.score(entity_label, label)
                score_ctx_bi = context_scorer_bi.score(context_text, comment)
                score_pop = pop_scores[i]
                score_pos_raw = i + 1

                # Type embedding scores
                score_types_embedding_basic = basic_types_embedding_scorer.score_types_embedding(entity_probabilities, ontology_types)
                score_types_embedding_topk = topk_types_embedding_scorer.score_types_embedding(entity_probabilities, ontology_types)
                score_types_embedding_maxner = max_ner_types_embedding_scorer.score_types_embedding(entity_probabilities, ontology_types)

                scored_candidates_local.append({
                    "context_text": context_text,
                    "entity_label": entity_label,
                    "start_position": start,
                    "end_position": end,
                    "candidate_label": label,
                    "candidate_uri": candidate.get("uri"),
                    "score_levenshtein": score_lev,
                    "score_context_bi": score_ctx_bi,
                    "score_popularity": score_pop,
                    "score_position_raw": score_pos_raw,
                    "score_types_embedding_basic_raw": score_types_embedding_basic,
                    "score_types_embedding_topk_raw": score_types_embedding_topk,
                    "score_types_embedding_maxner_raw": score_types_embedding_maxner
                })

                all_raw_pos_scores_local.append(score_pos_raw)
                all_raw_types_embedding_scores_local_basic.append(score_types_embedding_basic)
                all_raw_types_embedding_scores_local_topk.append(score_types_embedding_topk)
                all_raw_types_embedding_scores_local_maxner.append(score_types_embedding_maxner)

            # Normalization
            normalized_pos_scores = position_scorer.normalize_score(all_raw_pos_scores_local)
            normalized_types_embedding_scores_basic = basic_types_embedding_scorer.normalize_score(all_raw_types_embedding_scores_local_basic)
            normalized_types_embedding_scores_topk = topk_types_embedding_scorer.normalize_score(all_raw_types_embedding_scores_local_topk)
            normalized_types_embedding_scores_maxner = max_ner_types_embedding_scorer.normalize_score(all_raw_types_embedding_scores_local_maxner)

            for i in range(len(scored_candidates_local)):
                scored_candidates_local[i]["score_position"] = normalized_pos_scores[i]
                scored_candidates_local[i]["score_types_embedding_basic"] = normalized_types_embedding_scores_basic[i]
                scored_candidates_local[i]["score_types_embedding_topk"] = normalized_types_embedding_scores_topk[i]
                scored_candidates_local[i]["score_types_embedding_maxner"] = normalized_types_embedding_scores_maxner[i]

            scored_candidates.extend(scored_candidates_local)

    return scored_candidates


In [75]:
for model_name in loader.get_all_models():
    config = loader.get_config(model_name)
    ner_anns = config.named_entities_annotations
    pred_anns = config.predictions

    if len(ner_anns) != len(pred_anns):
        print(f"Error: Number of annotations and predictions do not match for {model_name}. Skipping.")
        continue

    print(f"\n--- Processing model: {model_name} ---")

    # === Initialize scorers dynamically based on model name ===
    basic_types_embedding_scorer = TypesEmbeddingScorer(ner_model_name=model_name)
    topk_types_embedding_scorer = TopKTypesEmbeddingScorer(ner_model_name=model_name)
    max_ner_types_embedding_scorer = MaxNERTypesEmbeddingScorer(ner_model_name=model_name)

    # Make them available globally for process_entry
    globals()["basic_types_embedding_scorer"] = basic_types_embedding_scorer
    globals()["topk_types_embedding_scorer"] = topk_types_embedding_scorer
    globals()["max_ner_types_embedding_scorer"] = max_ner_types_embedding_scorer

    scored_data = []

    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = []
        for i in range(len(ner_anns)):
            future = executor.submit(process_entry, ner_anns[i], pred_anns[i])
            futures.append(future)

        for future in tqdm(as_completed(futures), total=len(ner_anns), desc=f"Processing {model_name}"):
            try:
                scored_data.extend(future.result())
            except Exception as e:
                print(f"Error in thread: {e}")

    # === Save to CSV ===
    df = pd.DataFrame(scored_data)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_filename = f"scored_data_{model_name}_normalized_{timestamp}.csv"
    df.to_csv(csv_filename, index=False)
    print(f"Saved {model_name} results to {csv_filename}")



--- Processing model: conll ---


Processing conll: 100%|██████████| 940/940 [23:18<00:00,  1.49s/it]  


Saved conll results to scored_data_conll_normalized_20250423_181254.csv

--- Processing model: ontonotes ---


Processing ontonotes: 100%|██████████| 940/940 [14:39<00:00,  1.07it/s]


Saved ontonotes results to scored_data_ontonotes_normalized_20250423_182746.csv

--- Processing model: fewnerd ---


Processing fewnerd: 100%|██████████| 940/940 [14:06<00:00,  1.11it/s]


Saved fewnerd results to scored_data_fewnerd_normalized_20250423_184204.csv


#### Read scored candidates from csv 

In [40]:
df = pd.read_csv("scored_data_first_document_entities_normalized_20250422_222248.csv")

In [24]:
df.describe()

,start_position,end_position,score_levenshtein,score_context_bi,score_popularity,score_position_raw,score_types_embedding_basic_raw,score_types_embedding_topk_raw,score_types_embedding_maxner_raw,score_position,score_types_embedding_basic,score_types_embedding_topk,score_types_embedding_maxner
count,111596.000000,111596.000000,111596.000000,111596.000000,111596.000000,111596.000000,111596.000000,111596.000000,111596.000000,111596.000000,111596.000000,111596.000000,111596.000000
mean,397.458690,406.259956,0.462639,0.205012,0.378618,5.498862,0.095826,0.155221,0.155596,0.500000,0.473934,0.505457,0.505028
std,307.597795,307.710005,0.274474,0.177460,0.338898,2.872363,0.085584,0.132902,0.133105,0.319284,0.397048,0.423907,0.423355
min,0.000000,3.000000,0.000000,-0.208000,0.000000,1.000000,-0.069199,-0.053941,-0.053941,0.000000,0.000000,0.000000,0.000000
25%,128.000000,137.000000,0.240000,0.056000,0.075000,3.000000,0.000000,0.000000,0.000000,0.222000,0.000000,0.000000,0.000000
50%,339.000000,348.000000,0.445000,0.188000,0.286000,5.000000,0.093076,0.174142,0.176049,0.500000,0.511000,0.459000,0.459000
75%,612.000000,622.000000,0.640000,0.334000,0.635000,8.000000,0.154624,0.241450,0.241533,0.778000,0.889000,0.994000,0.994000
max,1391.000000,1399.000000,1.000000,0.843000,1.000000,10.000000,0.428192,0.546081,0.546084,1.000000,1.000000,1.000000,1.000000


In [25]:
df

,context_text,entity_label,start_position,end_position,candidate_label,candidate_uri,score_levenshtein,score_context_bi,score_popularity,score_position_raw,score_types_embedding_basic_raw,score_types_embedding_topk_raw,score_types_embedding_maxner_raw,score_position,score_types_embedding_basic,score_types_embedding_topk,score_types_embedding_maxner
0,China says time right for Taiwan talks . BEIJI...,China,0,5,China,http://dbpedia.org/resource/China,1.00,0.294,1.000,1,0.164444,0.218808,0.218836,1.000,1.000,0.909,0.909
1,China says time right for Taiwan talks . BEIJI...,China,0,5,Taiwan,http://dbpedia.org/resource/Taiwan,0.36,0.425,0.732,2,0.164444,0.218808,0.218836,0.889,1.000,0.909,0.909
2,China says time right for Taiwan talks . BEIJI...,China,0,5,China national football team,http://dbpedia.org/resource/China_national_foo...,0.30,0.281,0.259,3,0.092457,0.240767,0.240793,0.778,0.562,1.000,1.000
3,China says time right for Taiwan talks . BEIJI...,China,0,5,South China AA,http://dbpedia.org/resource/South_China_AA,0.53,0.154,0.140,4,0.092457,0.240767,0.240793,0.667,0.562,1.000,1.000
4,China says time right for Taiwan talks . BEIJI...,China,0,5,District (China),http://dbpedia.org/resource/District_(China),0.48,0.149,0.136,5,0.000000,0.000000,0.000000,0.556,0.000,0.000,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111591,GOLF - BRITISH MASTERS THIRD ROUND SCORES . NO...,Ireland,845,852,Dublin,http://dbpedia.org/resource/Dublin,0.31,-0.019,0.407,6,0.155025,0.239184,0.239204,0.444,0.954,1.000,1.000
111592,GOLF - BRITISH MASTERS THIRD ROUND SCORES . NO...,Ireland,845,852,Northern Ireland national football team,http://dbpedia.org/resource/Northern_Ireland_n...,0.30,-0.020,0.060,7,0.091277,0.237748,0.237748,0.333,0.562,0.994,0.994
111593,GOLF - BRITISH MASTERS THIRD ROUND SCORES . NO...,Ireland,845,852,Republic of Ireland national football team,http://dbpedia.org/resource/Republic_of_Irelan...,0.29,-0.033,0.054,8,0.091277,0.237748,0.237748,0.222,0.562,0.994,0.994
111594,GOLF - BRITISH MASTERS THIRD ROUND SCORES . NO...,Ireland,845,852,Republic of Ireland national under-21 football...,http://dbpedia.org/resource/Republic_of_Irelan...,0.24,0.019,0.000,9,0.091277,0.237748,0.237748,0.111,0.562,0.994,0.994


In [26]:
df_with_best_flag = create_best_candidate_flag_grouped(df.copy(), golden_annotations)


In [27]:
df_with_best_flag.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 111596 entries, 0 to 111595
Data columns (total 18 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   context_text                      111596 non-null  object 
 1   entity_label                      111596 non-null  object 
 2   start_position                    111596 non-null  int64  
 3   end_position                      111596 non-null  int64  
 4   candidate_label                   111596 non-null  object 
 5   candidate_uri                     111596 non-null  object 
 6   score_levenshtein                 111596 non-null  float64
 7   score_context_bi                  111596 non-null  float64
 8   score_popularity                  111596 non-null  float64
 9   score_position_raw                111596 non-null  int64  
 10  score_types_embedding_basic_raw   111596 non-null  float64
 11  score_types_embedding_topk_raw    111596 non-null  f

#### Feature selection

In [37]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# === 1. Build a grouped dataset for Random Forest ===
def build_grouped_dataframe_rf(df, selected_metrics, max_candidates=10):
    grouped_features = []
    targets = []

    for key, group in df.groupby(['context_text', 'entity_label', 'start_position', 'end_position']):
        try:
            candidates = group[[f'score_{m}' for m in selected_metrics]].values
        except KeyError as e:
            raise KeyError(f"Missing one or more selected features in the DataFrame: {e}")

        if len(candidates) < max_candidates:
            pad = np.zeros((max_candidates - len(candidates), candidates.shape[1]))
            candidates = np.vstack([candidates, pad])
        elif len(candidates) > max_candidates:
            candidates = candidates[:max_candidates]

        best_idx = group['is_best_candidate'].values.argmax()
        grouped_features.append(candidates.flatten().astype(np.float32))
        targets.append(best_idx)

    X = np.stack(grouped_features)
    y = np.array(targets)
    return X, y

# === 2. Define Feature Groups ===
baseline_metrics = ['levenshtein', 'context_bi', 'popularity', 'position']
embedding_extensions = ['types_embedding_basic', 'types_embedding_topk', 'types_embedding_maxner']

# === 3. Split Dataset into Train and Validation ===
df_full = df_with_best_flag
group_keys = ['context_text', 'entity_label', 'start_position', 'end_position']
unique_groups = df_full[group_keys].drop_duplicates()

train_keys, val_keys = train_test_split(unique_groups, test_size=0.2, random_state=42)

def filter_by_keys(df, keys_df):
    merged = df.merge(keys_df, on=group_keys, how='inner')
    return merged

df_train_split = filter_by_keys(df_full, train_keys)
df_val_split = filter_by_keys(df_full, val_keys)

# === 4. Evaluate Baseline and Embedding Extensions ===
results = []

def evaluate_feature_set(metrics_set, label):
    try:
        X_train_rf, y_train_rf = build_grouped_dataframe_rf(df_train_split, selected_metrics=metrics_set)
        X_val_rf, y_val_rf = build_grouped_dataframe_rf(df_val_split, selected_metrics=metrics_set)
    except KeyError as e:
        print(f"⚠️ Skipping {label} due to missing features: {e}")
        return

    rf_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', RandomForestClassifier(n_jobs=-1))
    ])

    rf_pipeline.fit(X_train_rf, y_train_rf)
    y_pred_val = rf_pipeline.predict(X_val_rf)

    acc = accuracy_score(y_val_rf, y_pred_val)
    report = classification_report(y_val_rf, y_pred_val, digits=4)

    results.append({
        'label': label,
        'metrics': metrics_set,
        'accuracy': acc,
        'report': report
    })

    print(f"✅ {label}: Validation Accuracy = {acc:.4f}\n")

# Baseline
print("\n🚀 Evaluating Baseline:")
evaluate_feature_set(baseline_metrics, label='Baseline (4 basic features)')

# Each embedding extension
for emb_metric in embedding_extensions:
    extended_metrics = baseline_metrics + [emb_metric]
    print(f"\n➕ Evaluating Extension with: {emb_metric}")
    evaluate_feature_set(extended_metrics, label=f'Baseline + {emb_metric}')

# === 5. Summary of Results ===
print("\n📊 Summary of Results:\n")
for res in results:
    print(f"{res['label']}:")
    print(f"  → Accuracy: {res['accuracy']:.4f}")
    print(f"  → Features: {res['metrics']}")
    print("  → Report:")
    print(res['report'])
    print("-" * 60)



🚀 Evaluating Baseline:
✅ Baseline (4 basic features): Validation Accuracy = 0.9390


➕ Evaluating Extension with: types_embedding_basic
✅ Baseline + types_embedding_basic: Validation Accuracy = 0.9403


➕ Evaluating Extension with: types_embedding_topk
✅ Baseline + types_embedding_topk: Validation Accuracy = 0.9430


➕ Evaluating Extension with: types_embedding_maxner
✅ Baseline + types_embedding_maxner: Validation Accuracy = 0.9399


📊 Summary of Results:

Baseline (4 basic features):
  → Accuracy: 0.9390
  → Features: ['levenshtein', 'context_bi', 'popularity', 'position']
  → Report:
              precision    recall  f1-score   support

           0     0.9387    0.9880    0.9627      1750
           1     0.9333    0.7609    0.8383       184
           2     0.9432    0.8058    0.8691       103
           3     1.0000    0.6939    0.8193        49
           4     0.9394    0.7561    0.8378        41
           5     0.9333    0.7778    0.8485        36
           6     1.0000   

In [38]:
df_full.describe()

,start_position,end_position,score_levenshtein,score_context_bi,score_popularity,score_position_raw,score_types_embedding_basic_raw,score_types_embedding_topk_raw,score_types_embedding_maxner_raw,score_position,score_types_embedding_basic,score_types_embedding_topk,score_types_embedding_maxner
count,111596.000000,111596.000000,111596.000000,111596.000000,111596.000000,111596.000000,111596.000000,111596.000000,111596.000000,111596.000000,111596.000000,111596.000000,111596.000000
mean,397.458690,406.259956,0.462639,0.205012,0.378618,5.498862,0.095826,0.155221,0.155596,0.500000,0.473934,0.505457,0.505028
std,307.597795,307.710005,0.274474,0.177460,0.338898,2.872363,0.085584,0.132902,0.133105,0.319284,0.397048,0.423907,0.423355
min,0.000000,3.000000,0.000000,-0.208000,0.000000,1.000000,-0.069199,-0.053941,-0.053941,0.000000,0.000000,0.000000,0.000000
25%,128.000000,137.000000,0.240000,0.056000,0.075000,3.000000,0.000000,0.000000,0.000000,0.222000,0.000000,0.000000,0.000000
50%,339.000000,348.000000,0.445000,0.188000,0.286000,5.000000,0.093076,0.174142,0.176049,0.500000,0.511000,0.459000,0.459000
75%,612.000000,622.000000,0.640000,0.334000,0.635000,8.000000,0.154624,0.241450,0.241533,0.778000,0.889000,0.994000,0.994000
max,1391.000000,1399.000000,1.000000,0.843000,1.000000,10.000000,0.428192,0.546081,0.546084,1.000000,1.000000,1.000000,1.000000


#### Train 

In [39]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Assuming df_with_best_flag is your DataFrame with the 'is_best_candidate' column
# created using the create_best_candidate_flag_grouped function

# === 1. Build a grouped dataset for Random Forest ===
def build_grouped_dataframe_rf(df, max_candidates=10):
    grouped_features = []
    targets = []

    # group by unique entity occurrence
    for key, group in df.groupby(['context_text','entity_label','start_position','end_position']):
        candidates = group[[
            'score_levenshtein',
            'score_context_bi',
            'score_position',
            'score_popularity',
            'score_types_embedding_basic'
        ]].values
        # pad up to max_candidates
        if len(candidates) < max_candidates:
            pad = np.zeros((max_candidates - len(candidates), candidates.shape[1]))
            candidates = np.vstack([candidates, pad])
        elif len(candidates) > max_candidates:
            candidates = candidates[:max_candidates]

        # find the index of the best
        best_idx = group['is_best_candidate'].values.argmax()  # first True
        grouped_features.append(candidates.flatten().astype(np.float32)) # Flatten for RF
        targets.append(best_idx)

    X = np.stack(grouped_features)      # (N_entities, 10 * 7)
    y = np.array(targets)                # (N_entities,)
    return X, y

X_rf, y_rf = build_grouped_dataframe_rf(df_with_best_flag, max_candidates=10)
print("X_rf shape:", X_rf.shape, "y_rf shape:", y_rf.shape)

# === 2. Train/Val Split for Random Forest ===
X_train_rf, X_val_rf, y_train_rf, y_val_rf = train_test_split(X_rf, y_rf, test_size=0.2) # Added random_state for reproducibility

# === 3. Define the Random Forest Pipeline ===
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),      # Feature scaling is often beneficial for RF
    ('classifier', RandomForestClassifier()) # You can add hyperparameters here
])

# === 4. Training the Random Forest Model ===
rf_pipeline.fit(X_train_rf, y_train_rf)

# === 5. Evaluation on Validation Set ===
y_pred_val_rf = rf_pipeline.predict(X_val_rf)
val_accuracy_rf = accuracy_score(y_val_rf, y_pred_val_rf)
print(f"\nValidation Accuracy (Random Forest): {val_accuracy_rf:.4f}")
print("\nValidation Classification Report (Random Forest):\n", classification_report(y_val_rf, y_pred_val_rf, zero_division=0))

# === 6. Feature Importance Assessment ===
feature_names_rf = [
    f"cand_{i}_score_levenshtein" for i in range(10)
] + [
    f"cand_{i}_score_context_bi" for i in range(10)
] + [
    f"cand_{i}_score_position" for i in range(10)
] + [
    f"cand_{i}_score_popularity" for i in range(10)
] + [
    f"cand_{i}_score_types_embedding" for i in range(10) # Added types embedding
]

if hasattr(rf_pipeline.named_steps['classifier'], 'feature_importances_'):
    importances = rf_pipeline.named_steps['classifier'].feature_importances_
    feature_importance_dict = dict(zip(feature_names_rf, importances))
    sorted_feature_importance = sorted(feature_importance_dict.items(), key=lambda item: item[1], reverse=True)

    print("\nFeature Importance (Random Forest):")
    for feature, importance in sorted_feature_importance:
        print(f"{feature}: {importance:.4f}")

    # Identify potentially less important metrics (across all candidates)
    importance_per_metric = {}
    metrics = [
        'levenshtein', 'context_bi', 'position', 'popularity', 'types_embedding'
    ]
    for metric in metrics:
        total_importance = 0
        for i in range(10):
            total_importance += feature_importance_dict.get(f"cand_{i}_score_{metric}", 0)
        importance_per_metric[metric] = total_importance

    sorted_metric_importance = sorted(importance_per_metric.items(), key=lambda item: item[1], reverse=True)
    print("\nTotal Importance per Metric:")
    for metric, total_importance in sorted_metric_importance:
        print(f"{metric}: {total_importance:.4f}")

    # Suggest dropping less important metrics (you'll need to define a threshold)
    threshold = 0.01 # Example threshold - adjust as needed
    less_important_metrics = [metric for metric, importance in sorted_metric_importance if importance < threshold]
    if less_important_metrics:
        print(f"\nPotentially less important metrics (below threshold {threshold}): {less_important_metrics}")
        print("Consider removing these metrics from your feature set and retraining.")
    else:
        print("\nAll metrics appear to have some level of importance based on the current model.")
else:
    print("\nFeature importance is not available for this Random Forest model.")

    # === 7. Evaluation on Test Set (similar to validation) ===

# First, re-split data: 80% train_val, 20% test
X_temp_rf, X_test_rf, y_temp_rf, y_test_rf = train_test_split(X_rf, y_rf, test_size=0.2, random_state=42)

# Then split train_val into train and val (e.g., 80% train, 20% val of 80%)
X_train_rf, X_val_rf, y_train_rf, y_val_rf = train_test_split(X_temp_rf, y_temp_rf, test_size=0.25, random_state=42)
# This results in: 60% train, 20% val, 20% test

# Re-train the model on training split (optional if you're using a single training phase)
rf_pipeline.fit(X_train_rf, y_train_rf)

# Predict on test set
y_pred_test_rf = rf_pipeline.predict(X_test_rf)

# Evaluate test accuracy and classification report
test_accuracy_rf = accuracy_score(y_test_rf, y_pred_test_rf)
print(f"\nTest Accuracy (Random Forest): {test_accuracy_rf:.4f}")
print("\nTest Classification Report (Random Forest):\n", classification_report(y_test_rf, y_pred_test_rf, zero_division=0))


X_rf shape: (11138, 50) y_rf shape: (11138,)

Validation Accuracy (Random Forest): 0.9349

Validation Classification Report (Random Forest):
               precision    recall  f1-score   support

           0       0.93      0.99      0.96      1707
           1       0.95      0.76      0.84       209
           2       1.00      0.76      0.86       125
           3       1.00      0.61      0.76        51
           4       0.97      0.76      0.85        38
           5       0.87      0.68      0.76        38
           6       1.00      0.76      0.86        25
           7       1.00      0.86      0.92         7
           8       0.92      0.92      0.92        13
           9       0.85      0.73      0.79        15

    accuracy                           0.93      2228
   macro avg       0.95      0.78      0.85      2228
weighted avg       0.94      0.93      0.93      2228


Feature Importance (Random Forest):
cand_1_score_levenshtein: 0.0616
cand_6_score_levenshtein: 0.0

#### Evaluation with test dataset

##### Score candidates

In [ ]:
import json
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Assuming you have your trained rf_pipeline object
# Also assuming you have initialized popularity_scorer, position_scorer,
# levenshtein_scorer, context_scorer_bi, context_scorer_cross,
# and types_embedding_scorer

# 1. Load the Evaluation Data (similar structure to training data)
evaluation_file_paths = [
    "../DjangoApp/NEL_project/NEL_app/Evaluation/candidate_selection_data/candidates_data_collector_20250422_162357.json", # test aida with NER ontonotes
    # Add more evaluation file paths if needed
]


# Load the JSON data (and merge these three files)
all_data = []
for json_file_path in evaluation_file_paths:
    with open(json_file_path, "r", encoding="utf-8") as file:
        all_data.append(json.load(file))


named_entities_annotations = all_data[0].get("named_entities_annotations", [])
predictions_data = all_data[0].get("predictions", [])



# Parallel processing for the first document's entities
num_documents_to_process = len(named_entities_annotations)
scored_data = []

if len(named_entities_annotations) == len(predictions_data):
    with ThreadPoolExecutor(max_workers=4) as executor: # Adjust max_workers as needed
        futures = []
        for i in range(min(num_documents_to_process, len(named_entities_annotations))):
            ner_annotation = named_entities_annotations[i]
            prediction_entry = predictions_data[i]
            future = executor.submit(process_entry, ner_annotation, prediction_entry)
            futures.append(future)

        for future in tqdm(as_completed(futures), total=num_documents_to_process, desc="Processing documents"):
            scored_data.extend(future.result())

else:
    print("Error: The number of named entity annotations and predictions does not match.")

# ========== Convert to DataFrame for Analysis or Model Input ==========
evaluation_df = pd.DataFrame(scored_data)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_filename = f'scored_data_first_document_entities_normalized_{timestamp}.csv'
evaluation_df.to_csv(csv_filename, index=False)
print(f"DataFrame saved to {csv_filename}")


Processing documents:   0%|          | 0/230 [00:00<?, ?it/s]

In [ ]:
import pandas as pd
from datetime import datetime

# Assuming your DataFrame is named 'df'
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save to a CSV file with timestamp in the filename
csv_filename = f'test_scored_data_{timestamp}.csv'
evaluation_df.to_csv(csv_filename, index=False)
print(f"DataFrame saved to {csv_filename}")

##### Read scored candidates from csv

In [41]:
evaluation_df = pd.read_csv("test_scored_data_20250422_232829.csv")
evaluation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27381 entries, 0 to 27380
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   context_text               27381 non-null  object 
 1   entity_label               27381 non-null  object 
 2   start_position             27381 non-null  int64  
 3   end_position               27381 non-null  int64  
 4   candidate_label            27377 non-null  object 
 5   candidate_uri              27381 non-null  object 
 6   score_levenshtein          27381 non-null  float64
 7   score_context_bi           27381 non-null  float64
 8   score_context_cross_raw    27381 non-null  float64
 9   score_jaccard              27381 non-null  float64
 10  score_popularity           27381 non-null  float64
 11  score_position_raw         27381 non-null  int64  
 12  score_types_embedding_raw  27381 non-null  float64
 13  score_context_cross        27381 non-null  flo

In [42]:
evaluation_df.describe()

,start_position,end_position,score_levenshtein,score_context_bi,score_context_cross_raw,score_jaccard,score_popularity,score_position_raw,score_types_embedding_raw,score_context_cross,score_position,score_types_embedding
count,27381.000000,27381.000000,27381.000000,27381.000000,27381.000000,27381.000000,27381.000000,27381.000000,27381.000000,27381.000000,27381.000000,27381.000000
mean,387.356853,396.193967,0.456702,0.206671,-4.950597,0.334608,0.377766,5.499836,0.103900,0.469175,0.500018,0.486978
std,301.464295,301.532923,0.273542,0.172528,2.559712,0.290729,0.338756,2.872410,0.091694,0.330276,0.319273,0.398555
min,0.000000,3.000000,0.000000,-0.170000,-11.323305,0.000000,0.000000,1.000000,-0.053733,0.000000,0.000000,0.000000
25%,134.000000,142.000000,0.240000,0.067000,-6.723903,0.000000,0.074000,3.000000,0.000000,0.182000,0.222000,0.000000
50%,328.000000,337.000000,0.440000,0.192000,-5.011048,0.333000,0.287000,5.000000,0.098932,0.447000,0.556000,0.544000
75%,590.000000,598.000000,0.640000,0.319000,-3.177863,0.500000,0.638000,8.000000,0.157462,0.744000,0.778000,0.893000
max,1325.000000,1335.000000,1.000000,0.811000,4.131427,1.000000,1.000000,10.000000,0.466345,1.000000,1.000000,1.000000


In [43]:

# 2. Load Golden Annotations for Evaluation
evaluation_golden_annotations_file = "../DjangoApp/NEL_project/NEL_app/Evaluation/candidate_selection_data/aida_test_golden_annotations_with_candidates.json" # Your updated path
with open(evaluation_golden_annotations_file, "r", encoding="utf-8") as f:
    evaluation_golden_annotations_data = json.load(f)
    evaluation_golden_annotations = evaluation_golden_annotations_data.get("golden_annotations", [])

# 3. Annotate Evaluation DataFrame with is_best_candidate flag
def create_best_candidate_flag_grouped_evaluation(predictions_df, golden_annotations):
    best_candidate_uris = {}
    for gold_entry in golden_annotations:
        if "entities" in gold_entry and isinstance(gold_entry["entities"], list):
            for ent in gold_entry["entities"]:
                if isinstance(ent, dict) and all(key in ent for key in ["entity_label", "start_position", "end_position", "best_candidate_uri"]):
                    key = (
                        gold_entry["content"],
                        ent["entity_label"],
                        ent["start_position"],
                        ent["end_position"],
                    )
                    best_candidate_uris[key] = ent["best_candidate_uri"]
                else:
                    print(f"Warning: Skipping malformed entity in golden annotation: {ent}")
        else:
            print(f"Warning: 'entities' key not found or is not a list in golden annotation entry: {gold_entry}")

    predictions_df["is_best_candidate"] = False
    for idx, row in predictions_df.iterrows():
        key = (
            row["context_text"],
            row["entity_label"],
            row["start_position"],
            row["end_position"],
        )
        if row["candidate_uri"] == best_candidate_uris.get(key):
            predictions_df.at[idx, "is_best_candidate"] = True
    return predictions_df

evaluation_df_with_best_flag = create_best_candidate_flag_grouped_evaluation(
    evaluation_df.copy(), evaluation_golden_annotations
)

# 4. Build Grouped Arrays for Evaluation
def build_grouped_dataframe_rf_evaluation(df, max_candidates=10):
    grouped_features = []
    targets = []

    for key, group in df.groupby(['context_text','entity_label','start_position','end_position']):
        candidates = group[[
            'score_levenshtein',
            'score_context_bi',
            'score_position',
            'score_popularity',
            'score_types_embedding'
        ]].values
        if len(candidates) < max_candidates:
            pad = np.zeros((max_candidates - len(candidates), candidates.shape[1]))
            candidates = np.vstack([candidates, pad])
        elif len(candidates) > max_candidates:
            candidates = candidates[:max_candidates]

        best_idx = group['is_best_candidate'].values.argmax() if any(group['is_best_candidate']) else 0
        grouped_features.append(candidates.flatten().astype(np.float32))
        targets.append(best_idx)

    X_eval_rf = np.stack(grouped_features)
    y_eval_rf = np.array(targets, dtype=np.int64)
    return X_eval_rf, y_eval_rf

X_eval_rf, y_eval_rf = build_grouped_dataframe_rf_evaluation(
    evaluation_df_with_best_flag, max_candidates=10
)
print("Evaluation X shape:", X_eval_rf.shape, "Evaluation y shape:", y_eval_rf.shape)

# 5. Evaluate the Trained Model
y_pred_eval_rf = rf_pipeline.predict(X_eval_rf)
eval_accuracy_rf = accuracy_score(y_eval_rf, y_pred_eval_rf)
print(f"\nEvaluation Accuracy (Random Forest): {eval_accuracy_rf:.4f}")
print("\nEvaluation Classification Report (Random Forest):\n", classification_report(y_eval_rf, y_pred_eval_rf, zero_division=0))

Evaluation X shape: (2732, 50) Evaluation y shape: (2732,)

Evaluation Accuracy (Random Forest): 0.8393

Evaluation Classification Report (Random Forest):
               precision    recall  f1-score   support

           0       0.84      0.99      0.91      2074
           1       0.92      0.48      0.63       232
           2       0.85      0.31      0.45       146
           3       0.81      0.34      0.48        88
           4       0.64      0.17      0.27        53
           5       0.50      0.16      0.24        25
           6       0.56      0.22      0.32        45
           7       1.00      0.31      0.48        32
           8       1.00      0.43      0.60        21
           9       0.80      0.25      0.38        16

    accuracy                           0.84      2732
   macro avg       0.79      0.37      0.48      2732
weighted avg       0.84      0.84      0.81      2732

